In [1]:
import numpy as np
import json

In [4]:
pwd

'c:\\Users\\jiang\\Desktop\\Work\\VisualizationOfThoughts\\reasoning-viz'

In [6]:
history = []
with open("runs/20260924-154029-840287.events.jsonl", "r") as f:
    for line in f:
        history.append(json.loads(line))

In [17]:
history[9]

{'recorded_at': '2026-09-24T19:40:35.046124+00:00',
 'record_type': 'sdk_notification',
 'payload': {'method': 'item/completed',
  'payload': {'completedAtMs': 1790278835026,
   'item': {'content': [],
    'id': 'rs_00f7b108e9e00c21016ab57cb0d1b487d1999c37122ae71da0',
    'summary': ['**Planning file inspection and weather research**'],
    'type': 'reasoning'},
   'threadId': '01a0d4ef-0bcc-77a1-b91c-e062680e33a7',
   'turnId': '01a0d4ef-0cd4-7ac0-8d86-06dd9bb451ac'}}}

In [19]:
import json
from pathlib import Path
from pprint import pprint

trace_path = Path("runs/20260924-154029-840287.events.jsonl")

with trace_path.open("r", encoding="utf-8") as f:
    history = [json.loads(line) for line in f if line.strip()]


extracted = {
    "summaries": [],
    "plans": [],
    "progress_updates": [],
    "function_calls": [],
    "function_results": [],
    "final_answer": None,
    "timeline": [],
}

# Item types representing tool/function activity.
call_types = {
    "commandExecution",
    "webSearch",
    "mcpToolCall",
    "dynamicToolCall",
    "functionCall",
    "computerToolCall",
}

result_types = {
    "functionCallOutput",
    "mcpToolCallOutput",
    "dynamicToolCallOutput",
}


for record in history:
    outer = record.get("payload", {})

    if record.get("record_type") != "sdk_notification":
        continue

    method = outer.get("method")
    event_payload = outer.get("payload", {})

    # Use completed items as the authoritative representation.
    if method == "item/completed":
        item = event_payload.get("item", {})
        item_type = item.get("type")
        phase = item.get("phase")
        text = item.get("text")

        if item_type == "reasoning":
            summaries = item.get("summary", [])

            if isinstance(summaries, str):
                summaries = [summaries]

            for summary in summaries:
                extracted["summaries"].append(summary)
                extracted["timeline"].append({
                    "category": "reasoning_summary",
                    "text": summary,
                    "item": item,
                })

        elif item_type == "plan":
            plan_text = text or item.get("plan")

            if plan_text:
                extracted["plans"].append(plan_text)
                extracted["timeline"].append({
                    "category": "plan",
                    "text": plan_text,
                    "item": item,
                })

        elif item_type == "agentMessage":
            if phase == "commentary":
                extracted["progress_updates"].append(text)
                category = "progress_update"

            elif phase == "final_answer":
                extracted["final_answer"] = text
                category = "final_answer"

            else:
                # Some providers omit phase. Treat the latest unphased
                # agent message as a possible final answer.
                extracted["final_answer"] = text
                category = "agent_message"

            extracted["timeline"].append({
                "category": category,
                "text": text,
                "phase": phase,
                "item": item,
            })

        elif item_type in call_types:
            extracted["function_calls"].append(item)
            extracted["timeline"].append({
                "category": "function_call",
                "tool_type": item_type,
                "item": item,
            })

        elif item_type in result_types:
            extracted["function_results"].append(item)
            extracted["timeline"].append({
                "category": "function_result",
                "tool_type": item_type,
                "item": item,
            })


# The wrapper also stores the final response in run_result.
if extracted["final_answer"] is None:
    for record in reversed(history):
        if record.get("record_type") == "run_result":
            extracted["final_answer"] = record.get(
                "payload", {}
            ).get("final_response")
            break


print("REASONING SUMMARIES")
pprint(extracted["summaries"])

print("\nPLANS")
pprint(extracted["plans"])

print("\nPROGRESS UPDATES")
pprint(extracted["progress_updates"])

print("\nFUNCTION / TOOL CALLS")
pprint(extracted["function_calls"])

print("\nFUNCTION RESULTS")
pprint(extracted["function_results"])

print("\nFINAL ANSWER")
print(extracted["final_answer"])

REASONING SUMMARIES
['**Planning file inspection and weather research**',
 '**Planning climate research for travel timing**',
 '**Searching WeatherSpark for mid-November data**',
 '**Clarifying volcano hike location and weather**',
 '**Clarifying northern Japan apple city ambiguity**',
 '**Compiling mid-November weather and daylight data**',
 '**Summarizing Antigua weather normals for recommendations**']

PLANS
[]

PROGRESS UPDATES
[]

FUNCTION / TOOL CALLS
[{'aggregatedOutput': 'City famous for eiffel tower\r\n'
                      'City where the famous VolcÃ¡n de Fuego hike is '
                      'located\r\n'
                      'City in northern part of Japan renowned for massive '
                      'apple production.\r\n',
  'command': '"C:\\\\Windows\\\\System32\\\\WindowsPowerShell\\\\v1.0\\\\powershell.exe" '
             '-Command "Get-Content -Raw -LiteralPath \'Cities.txt\'"',
  'commandActions': [{'command': "Get-Content -Raw -LiteralPath 'Cities.txt'",
       